In [ ]:
import os
import re
import json
import glob
import asyncio
import requests
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict
from tabulate import tabulate
from rank_bm25 import BM25Okapi
from time import perf_counter  # <<< ADD
import hashlib  # <<< ADD

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

import chromadb  # pure chroma
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama  # hanya LLM-nya aja

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# --- TOTAL TIMING HELPERS (ADD) ---
def _now() -> float:
    return perf_counter()

def _elapsed_ms(t0: float) -> int:
    return int((perf_counter() - t0) * 1000)

def _log_total(label: str, t0: float):
    ms = _elapsed_ms(t0)
    print(f"\n[TOTAL] {label}: {ms} ms (~{ms/1000:.3f}s)")
# ----------------------------------

# =========================
# === CONFIG (ENV) ========
# =========================
JSON_INPUT       = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR       = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME  = os.getenv("COLLECTION_NAME", "classicmodels")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL       = os.getenv("CHAT_MODEL", "HridaAI/hrida-t2sql:q8_0")
TOP_K            = int(os.getenv("TOP_K", "5"))

MICRO_PROBE = ["id", "date", "status"]

SPARSE_BONUS     = float(os.getenv("SPARSE_BONUS", "0.75"))
NAME_QUERY_BONUS = float(os.getenv("NAME_QUERY_BONUS", "0.3"))
ID_QUERY_BONUS   = float(os.getenv("ID_QUERY_BONUS", "0.4"))

# >>> SPEED: batasi berapa banyak dokumen yang masuk ke context LLM
MAX_CTX_DOCS = int(os.getenv("MAX_CTX_DOCS", "12"))
SPARSE_MAX_HITS  = int(os.getenv("SPARSE_MAX_HITS", "20"))

# bisa dimatiin kalau mau lebih silent
SPARSE_DEBUG   = bool(int(os.getenv("SPARSE_DEBUG", "0")))
RETRIEVE_DEBUG = bool(int(os.getenv("RETRIEVE_DEBUG", "1")))

COLUMN_PRIORITY_CONFIG: List[Dict[str, Any]] = [
    {
        "id": "customers.primary_name",
        "tables": ["customers"],
        "role_any": ["name"],
        "alias_any": [
            "customer name",
            "nama customer",
            "nama pelanggan",
            "company name",
            "nama perusahaan",
            "account name",
        ],
        "column_contains": ["customername"],
        "query_terms_any": ["name", "nama"],
        "query_terms_also_any": ["customer", "customers", "pelanggan"],
        "boost": 0.4,
    },
]

MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

SQL_DEBUG = int(os.getenv("SQL_DEBUG", "1"))
def _dbg_print_two(label: str, sql: str):
    if SQL_DEBUG:
        print(f"\n{label}")
        print(sql)

# =========================
# === RAG (STATE) =========
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str
    draft_sql: str
    final_sql: str
    llm_ms: int            # <<< ADD: total waktu LLM (ms)

# =========================
# === UTIL ================
# =========================
def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return sorted(out)

# --- FIXED: OllamaEmbeddingFunction (indent + clean config methods) ---
class OllamaEmbeddingFunction(EmbeddingFunction):
    def __init__(self, base_url: str, model: str = "bge-m3:latest", timeout: int = 60):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = int(timeout)

    def __call__(self, texts: Documents) -> Embeddings:
        out: Embeddings = []
        for t in texts:
            resp = requests.post(
                f"{self.base_url}/api/embeddings",
                json={"model": self.model, "prompt": t},
                timeout=self.timeout,
            )
            resp.raise_for_status()
            out.append(resp.json()["embedding"])
        return out

    @classmethod
    def name(cls) -> str:
        return "ollama"

    def get_config(self) -> Dict[str, Any]:
        return {
            "name": self.name(),
            "base_url": self.base_url,
            "model": self.model,
            "timeout": self.timeout,
        }

    @classmethod
    def build_from_config(cls, config: dict) -> "OllamaEmbeddingFunction":
        return cls(
            base_url=str(config.get("base_url", "http://localhost:11434")),
            model=str(config.get("model", "bge-m3:latest")),
            timeout=int(config.get("timeout", 60)),
        )

    # new: allow Chroma to serialize EF config
    def get_config(self) -> Dict[str, Any]:
        return {
            "name": self.name(),
            "base_url": self.base_url,
            "model": self.model,
            "timeout": self.timeout,
        }

def build_vector_store():
    os.makedirs(CHROMA_DIR, exist_ok=True)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    emb_fn = OllamaEmbeddingFunction(
        base_url=OLLAMA_BASE_URL,
        model="bge-m3:latest",
    )
    col = client.get_or_create_collection(
        COLLECTION_NAME,
        embedding_function=emb_fn,
    )
    return col

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

# --- Chroma metadata sanitizer (PENTING) ---
# --- Normalizers & volatile keys ---
VOLATILE_KEYS = {
    "_source", "updated_at", "lastModified", "timestamp", "ts"
}

def _norm_str(s: Any) -> str:
    s = "" if s is None else str(s)
    # trim & collapse spasi berlebih per baris, lalu strip
    s = "\n".join(re.sub(r"[ \t]{2,}", " ", line.rstrip()) for line in s.splitlines())
    return s.strip()

def _norm_list(x: Any) -> list[str]:
    # list stabil: trim, dedup case-insensitive, lalu sort case-insensitive
    if isinstance(x, str):
        try:
            j = json.loads(x)
            x = j if isinstance(j, list) else [x]
        except Exception:
            x = [x]
    elif not isinstance(x, list):
        x = [x]
    cleaned, seen = [], set()
    for v in x:
        s = _norm_str(v)
        k = s.lower()
        if s and k not in seen:
            seen.add(k)
            cleaned.append(s)
    return sorted(cleaned, key=str.lower)
    
def _sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hanya simpan tipe primitif. list/dict -> JSON string yang STABIL (sort_keys).
    Buang kunci yang volatile agar tidak memicu upsert palsu.
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if k in VOLATILE_KEYS:
            continue  # drop volatile keys
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[k] = _norm_str(v) if isinstance(v, str) else v
        elif isinstance(v, list):
            # stabilkan list sebelum dump
            vv = _norm_list(v)
            out[k] = json.dumps(vv, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        elif isinstance(v, dict):
            out[k] = json.dumps(v, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        else:
            out[k] = _norm_str(v)
    return out

# =========================
# === SPARSE (BM25) =======
# =========================
_BM25_CACHE: dict[str, Any] = {}
_STOPWORDS = {
    "the","a","an","to","of","and","or","for","on","in","at","by",
    "yang","dan","atau","ke","dari","di","itu","ini","with"
}

def _normalize_text(txt: str) -> str:
    txt = txt.lower()
    txt = re.sub(r"(\d)[,\.](\d{3})(\b|$)", r"\1\2", txt)
    txt = re.sub(r"(\w)'s\b", r"\1", txt)
    txt = txt.replace("'", "")
    txt = re.sub(r"[^a-z0-9_]+", " ", txt)
    return txt.strip()

def _tokenize_bm25_query(q: str) -> List[str]:
    s = _normalize_text(q)
    toks = [t for t in s.split() if t and t not in _STOPWORDS]
    return toks

def _jsonl_iter_records(jsonl_paths: List[str]) -> Iterable[Dict[str, Any]]:
    for path in jsonl_paths:
        p = Path(path)
        if not p.exists():
            continue
        for _, obj in read_jsonl(path):
            if isinstance(obj, dict):
                yield obj

def build_sparse_query_from_question(q: str) -> str:
    toks = _tokenize_bm25_query(q)
    return " ".join(toks)

def _doc_tokens_from_meta(meta: Dict[str, Any]) -> List[str]:
    # gunakan table, column, description, aliases/roles sebagai corpus
    table = str(meta.get("table") or meta.get("tableName") or "").lower()
    column = str(meta.get("column") or meta.get("columnName") or "").lower()
    desc = str(meta.get("description") or "").lower()
    aliases = meta.get("tags_alias") or meta.get("aliases") or []
    roles = meta.get("tags_role") or meta.get("roles") or []
    if isinstance(aliases, str):
        try:
            aliases = json.loads(aliases)
        except Exception:
            aliases = [aliases]
    if isinstance(roles, str):
        try:
            roles = json.loads(roles)
        except Exception:
            roles = [roles]
    bag = " ".join([table, column, desc] + [str(a) for a in (aliases or [])] + [str(r) for r in (roles or [])])
    return _tokenize_bm25_query(bag)

def jsonl_sparse_search(q: str, jsonl_paths: List[str], max_hits: int = 20) -> List[str]:
    """
    Kembalikan daftar candidate IDs (field_urn/urn fallback f"{source}:{table}::{column}") berdasar BM25.
    Di-cache per (paths hash) agar cepat.
    """
    # cache key by files + mtime
    sig = []
    for p in jsonl_paths:
        try:
            st = os.stat(p)
            sig.append(f"{p}:{int(st.st_mtime)}:{int(st.st_size)}")
        except Exception:
            sig.append(p)
    cache_key = "||".join(sorted(sig))

    if cache_key not in _BM25_CACHE:
        docs: List[str] = []
        ids: List[str] = []

        for path in jsonl_paths:
            p = Path(path)
            if not p.exists():
                continue
            srcname = p.name

            for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
                # ---- CASE 1: line bukan dict → coba list-of-dict ----
                if not isinstance(obj, dict):
                    try:
                        maybe = json.loads(raw)
                    except Exception:
                        maybe = None

                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if not isinstance(item, dict):
                                continue
                            table = str(item.get("table") or item.get("tableName") or "").strip().lower()
                            column = str(item.get("column") or item.get("columnName") or "").strip().lower()

                            doc_id_raw = (
                                item.get("field_urn")
                                or item.get("urn")
                                or f"{srcname}:{table}::{column}"
                            )
                            doc_id = str(doc_id_raw)

                            docs.append(" ".join(_doc_tokens_from_meta(item)))
                            ids.append(doc_id)
                        continue  # lanjut line berikutnya
                    # kalau bukan dict & bukan list-of-dict → skip line
                    continue

                # ---- CASE 2: obj sudah dict normal ----
                table = str(obj.get("table") or obj.get("tableName") or "").strip().lower()
                column = str(obj.get("column") or obj.get("columnName") or "").strip().lower()

                doc_id_raw = (
                    obj.get("field_urn")
                    or obj.get("urn")
                    or f"{srcname}:{table}::{column}"
                )
                doc_id = str(doc_id_raw)

                docs.append(" ".join(_doc_tokens_from_meta(obj)))
                ids.append(doc_id)

        if not docs:
            _BM25_CACHE[cache_key] = {"bm25": None, "ids": []}
        else:
            tokenized_corpus = [d.split() for d in docs]
            _BM25_CACHE[cache_key] = {
                "bm25": BM25Okapi(tokenized_corpus),
                "ids": ids,
            }

    entry = _BM25_CACHE[cache_key]
    if not entry["bm25"]:
        return []

    query_tokens = _tokenize_bm25_query(q)
    scores = entry["bm25"].get_scores(query_tokens)
    ranked = sorted(zip(entry["ids"], scores), key=lambda x: x[1], reverse=True)
    out = [doc_id for doc_id, _ in ranked[:max_hits]]

    if SPARSE_DEBUG:
        print("[SPARSE] top hits:")
        for doc_id, sc in ranked[:min(max_hits, 10)]:
            print(f"  - {doc_id}  score={sc:.3f}")

    return out

def _doc_matches_sparse(md: Dict[str, Any], sparse_ids: List[str]) -> bool:
    if not md or not sparse_ids:
        return False

    # samakan case untuk matching
    sid_lower = {x.lower() for x in sparse_ids}

    urn = str(md.get("field_urn") or md.get("urn") or "").lower()
    if urn and urn in sid_lower:
        return True

    table = str(md.get("table") or md.get("tableName") or "").strip().lower()
    column = str(md.get("column") or md.get("columnName") or "").strip().lower()
    if table and column:
        suffix = f":{table}::{column}"
        # cek suffix juga dalam lowercase
        return any(x.lower().endswith(suffix) for x in sparse_ids)

    return False

# =========================
# === DENSE PROBE ========
# =========================
def build_dynamic_probe(question: str, max_tokens: int = 28) -> str:
    q = (question or "").lower()

    def _mentions_id_like(text: str) -> bool:
        if re.search(r"\b(id|number|nomor|no|code|kode)\b", text):
            return True
        if "customer number" in text or "customer_number" in text:
            return True
        return False

    identity_only = (
        any(k in q for k in ["name", "nama"])
        and not any(k in q for k in [
            "sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net",
            "count", "berapa banyak", "how many", "hitung",
            "avg", "average", "mean", "rata", "rata-rata",
            "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
            "month", "week", "year", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
        ])
    )

    base: List[str] = [] if identity_only else MICRO_PROBE[:]

    if any(k in q for k in ["avg", "average", "mean", "rata", "rata-rata"]):
        base += ["avg", "mean", "average", "metric", "measure", "price"]
    if any(k in q for k in ["sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net"]):
        base += ["sum", "total", "amount", "revenue", "sales", "price", "gross", "net"]
    if any(k in q for k in ["count", "berapa banyak", "how many", "hitung"]):
        base += ["count", "qty", "quantity", "number"]
    if any(k in q for k in [
        "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
        "month", "week", "year", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
    ]):
        base += ["date", "timestamp", "month", "week", "quarter", "year", "created", "updated", "order_date"]

    mentions_customer = any(k in q for k in ["customer", "customers", "pelanggan"])
    if mentions_customer:
        base += ["customer", "customer name"]
        if not _mentions_id_like(q):
            base += ["customerName"]

    if any(k in q for k in ["country", "negara", "city", "kota", "region", "wilayah"]):
        base += ["country", "city", "region"]
    if any(k in q for k in ["name", "nama"]):
        base += ["name"]
    if any(k in q for k in ["first name", "nama depan"]):
        base += ["first name", "contact first name"]
    if any(k in q for k in ["last name", "nama belakang", "surname"]):
        base += ["last name", "contact last name"]

    tokens = list(dict.fromkeys(base))[:max_tokens]

    if len(q.split()) <= 2:
        for t in ["amount", "price", "revenue", "month", "year"]:
            if t not in tokens and len(tokens) < max_tokens:
                tokens.append(t)

    probe_tail = " ".join(tokens).strip()
    return f"{question} {probe_tail}".strip()
    
# =========================
# === INGEST (PURE CHROMA)
# =========================
COMPARE_DEBUG = bool(int(os.getenv("COMPARE_DEBUG", "0")))

# kunci yang sering berubah & memicu upsert palsu → kita buang
VOLATILE_KEYS = {
    "_source", "last_ingested", "lastIngested", "ingested_at",
    "updated_at", "lastModified", "timestamp", "ts"
}

def _norm_str(s: Any) -> str:
    s = "" if s is None else str(s)
    # rapikan whitespace tanpa mengubah isi
    s = "\n".join(re.sub(r"[ \t]{2,}", " ", line.rstrip()) for line in s.splitlines())
    return s.strip()

def _norm_list(x: Any) -> List[str]:
    # list stabil: json->list bila string JSON, trim, dedup casefold, lalu sort
    if isinstance(x, str):
        try:
            j = json.loads(x)
            x = j if isinstance(j, list) else [x]
        except Exception:
            x = [x]
    elif not isinstance(x, list):
        x = [x]
    cleaned, seen = [], set()
    for v in x:
        s = _norm_str(v)
        k = s.casefold()
        if s and k not in seen:
            seen.add(k)
            cleaned.append(s)
    return sorted(cleaned, key=str.casefold)

def _stable_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

# --- NEW: koersi nilai meta agar perbandingan tipe-stabil ---
def _coerce_json_like(v: Any) -> Any:
    """
    - Jika v string yang tampak seperti JSON list/dict → parse.
    - List → normalisasi via _norm_list.
    - Dict → biarkan, tapi saat dump akan sort_keys=True.
    """
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                v = json.loads(s)
            except Exception:
                return v
        else:
            return v
    if isinstance(v, list):
        return _norm_list(v)
    return v

def _sanitize_for_compare(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Versi khusus untuk perbandingan: parse kembali string JSON agar
    list/dict dibandingkan sebagai struktur, bukan sebagai string mentah.
    Abaikan _canon_hash agar tidak circular-diff.
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if k in VOLATILE_KEYS or k == "_canon_hash":  # <<< ADD ignore hash on compare
            continue
        vv = _coerce_json_like(v)
        out[k] = vv
    return out

def _sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hanya simpan tipe primitif. list/dict → JSON string STABIL.
    Buang kunci volatile agar tidak memicu 'updated' palsu.
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if k in VOLATILE_KEYS:
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[k] = _norm_str(v) if isinstance(v, str) else v
        elif isinstance(v, list):
            out[k] = _stable_json(_norm_list(v))
        elif isinstance(v, dict):
            out[k] = _stable_json(v)
        else:
            out[k] = _norm_str(v)
    return out

def _canon_doc(s: str) -> str:
    # hilangkan trailing spaces per-baris agar stabil
    return "\n".join(line.rstrip() for line in (s or "").splitlines()).strip()

def _canon_meta(d: Dict[str, Any]) -> str:
    """
    Dump meta ke JSON STABIL setelah tipe dinormalkan agar list/dict vs string JSON
    tidak bikin beda palsu. Abaikan _canon_hash pada tahap compare.
    """
    try:
        normalized = _sanitize_for_compare(d or {})
        return _stable_json(normalized)
    except Exception:
        return str(d or {})

# <<< ADD: Hash kanonik untuk early-skip >>>
def _compute_canon_hash(doc: str, meta: Dict[str, Any]) -> str:
    """
    Hitung hash stabil dari dokumen & metadata kanonik.
    Metadata di-normalize via _canon_meta (yang mengabaikan _canon_hash).
    """
    a = _canon_doc(doc or "")
    b = _canon_meta(meta or {})
    payload = f"{a}\n{b}".encode("utf-8")
    return hashlib.sha1(payload).hexdigest()

def _print_meta_doc_diff(doc_id: str, old_doc: str, new_doc: str, old_meta: Dict[str, Any], new_meta: Dict[str, Any]):
    if not COMPARE_DEBUG:
        return
    a_doc, b_doc = _canon_doc(old_doc), _canon_doc(new_doc)
    a_meta, b_meta = _canon_meta(old_meta), _canon_meta(new_meta)
    if a_doc == b_doc and a_meta == b_meta:
        return
    print(f"[DIFF] {doc_id}")
    if a_doc != b_doc:
        print("  - documents differ")
    if a_meta != b_meta:
        try:
            om = json.loads(a_meta); nm = json.loads(b_meta)
            keys = sorted(set(om.keys()) | set(nm.keys()))
            for k in keys:
                if om.get(k) != nm.get(k):
                    print(f"  * meta[{k}]: {om.get(k)!r} -> {nm.get(k)!r}")
        except Exception:
            print("  - metadatas differ (unparseable)")
            
def _ingest_one_json(col, meta: dict, source_name: str, debug_id: str):
    """
    Upsert satu item ke koleksi Chroma.
    Return: (added, updated, skipped)
    """
    # 1) tentukan id
    doc_id = meta.get("field_urn") or meta.get("urn")
    if not doc_id:
        table_raw  = meta.get("table") or meta.get("tableName") or ""
        column_raw = meta.get("column") or meta.get("columnName") or ""

        table  = _norm_str(table_raw)
        column = _norm_str(column_raw)
        if not table and not column:
            return 0, 0, 1  # skip kalau minim info
        doc_id = f"{Path(source_name).name}:{table}::{column}".lower()

    # 2) normalisasi nilai penting
    aliases = _norm_list(meta.get("tags_alias") or meta.get("aliases") or [])
    roles   = _norm_list(meta.get("tags_role")  or meta.get("roles")   or [])
    table   = _norm_str(meta.get("table") or meta.get("tableName") or "")
    column  = _norm_str(meta.get("column") or meta.get("columnName") or "")
    desc    = _norm_str(meta.get("description") or "")

    # 3) dokumen teks STABIL
    new_doc = (
        f"table: {table}\n"
        f"column: {column}\n"
        f"description: {desc}\n"
        f"aliases: {', '.join(aliases)}\n"
        f"roles: {', '.join(roles)}"
    )

    # 4) metadata STABIL (tanpa _source & co), isi kolom kanonik
    meta2 = dict(meta or {})
    meta2["table"]       = table
    meta2["tableName"]   = table or meta2.get("tableName")
    meta2["column"]      = column
    meta2["columnName"]  = column or meta2.get("columnName")
    meta2["description"] = desc
    meta2["tags_alias"]  = aliases
    meta2["aliases"]     = aliases
    meta2["tags_role"]   = roles
    meta2["roles"]       = roles
    # JANGAN menyimpan _source agar tidak flip-flop
    if "_source" in meta2:
        del meta2["_source"]
    new_meta = _sanitize_metadata(meta2)

    # <<< ADD: hitung & tanam _canon_hash supaya run berikutnya bisa early-skip >>>
    canon_hash = _compute_canon_hash(new_doc, new_meta)
    new_meta["_canon_hash"] = canon_hash

    # 5) cek eksistensi
    try:
        got = col.get(ids=[doc_id], include=["documents", "metadatas"])
        exists = bool(got.get("ids"))
    except Exception:
        exists = False

    if not exists:
        col.add(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
        return 1, 0, 0

    # 6) bandingkan STABIL (tipe-agnostik) + early-skip

    # ambil dok & meta lama dari Chroma
    old_doc  = ((got.get("documents") or [""])[0]) if got else ""
    old_meta = ((got.get("metadatas") or [{}])[0]) if got else {}

    # --- EARLY 1: kalau last_ingested / lastIngested sama persis, langsung skip ---
    new_last = meta.get("last_ingested") or meta.get("lastIngested")
    old_last = old_meta.get("last_ingested") or old_meta.get("lastIngested")
    if new_last and old_last and str(new_last) == str(old_last):
        # ingestion timestamp sama → asumsi snapshot sama → nggak usah update
        return 0, 0, 1

    # --- EARLY 2: jika hash sama, langsung skip ---
    old_hash = str(old_meta.get("_canon_hash") or "").strip()
    if old_hash and old_hash == canon_hash:
        return 0, 0, 1

    # cek diff kanonik (doc + meta) sebagai safety net
    same_doc  = _canon_doc(old_doc)   == _canon_doc(new_doc)
    same_meta = _canon_meta(old_meta) == _canon_meta(new_meta)

    if same_doc and same_meta:
        # kalau hash lama belum ada, update sekali untuk “menanam” hash
        if not old_hash:
            col.update(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
            return 0, 1, 0
        return 0, 0, 1  # unchanged → skip

    # 7) ada beda → update + (opsional) tampilkan diff
    _print_meta_doc_diff(doc_id, old_doc, new_doc, old_meta, new_meta)
    col.update(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
    return 0, 1, 0

def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    col = build_vector_store()

    added = updated = skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
            if obj is None:
                try:
                    maybe = json.loads(raw)
                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if isinstance(item, dict):
                                _a, _u, _s = _ingest_one_json(
                                    col, item, p.name, f"{path}:{line_no}[{i}]"
                                )
                                added += _a
                                updated += _u
                                skipped += _s
                        continue
                except Exception:
                    pass
                print(f"[INGEST] skip {path}:{line_no} (not dict/list)")
                continue

            _a, _u, _s = _ingest_one_json(col, obj, p.name, f"{path}:{line_no}")
            added += _a
            updated += _u
            skipped += _s

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

# =========================
# === COLUMN PRIOR BOOST ==
# =========================
def apply_column_priority_boost(
    q_l: str,
    table: str,
    col_name: str,
    roles: List[str],
    aliases: List[str],
) -> float:
    bonus = 0.0
    if any(k in q_l for k in ["name", "nama"]):
        if any(r == "name" or "name" in r for r in roles) or "name" in col_name:
            bonus += NAME_QUERY_BONUS
    if table.lower() == "customers":
        mentions_customer = any(k in q_l for k in ["customer", "customers", "pelanggan"])
        asks_id_like = any(k in q_l for k in [" id", "id ", " id,", "identifier", "kode", "code", "customer number"])
        if mentions_customer and not asks_id_like:
            if "customername" in col_name or "customer_name" in col_name:
                bonus += 0.6
            if "customernumber" in col_name or col_name.endswith("number"):
                bonus -= 0.35
    if any(k in q_l for k in [" id ", "id ", " id,", "identifier", "kode", "code"]):
        if (
            any(r == "id" or "id" in r for r in roles)
            or "number" in col_name
            or col_name.lower().endswith("id")
        ):
            bonus += ID_QUERY_BONUS

    col_lower = col_name.lower()
    role_set = set(roles)
    alias_set = set(aliases)
    for rule in COLUMN_PRIORITY_CONFIG:
        tables = [t.lower() for t in rule.get("tables", []) or []]
        if tables and table.lower() not in tables:
            continue

        role_any = rule.get("role_any") or []
        if role_any and not any(r in role_set for r in role_any):
            continue

        alias_any = rule.get("alias_any") or []
        if alias_any and not any(a in alias_set for a in alias_any):
            continue

        column_contains = rule.get("column_contains") or []
        if column_contains and not any(sub in col_lower for sub in column_contains):
            continue

        qt_any = rule.get("query_terms_any") or []
        if qt_any and not any(t in q_l for t in qt_any):
            continue

        qt_also = rule.get("query_terms_also_any") or []
        if qt_also and not any(t in q_l for t in qt_also):
            continue

        bonus += float(rule.get("boost", 0.0))

    return bonus

# ============================================================
# === NEW: simple query classifier + table scorer ============
# ============================================================
def classify_query(q: str) -> str:
    """return: 'entity_list', 'entity_with_metric', or 'other'"""
    ql = q.lower()
    if any(kw in ql for kw in ["total", "count", "jumlah", "how many", "per ", "group by", "avg", "average", "sum"]):
        return "entity_with_metric"
    if any(kw in ql for kw in ["list", "show", "tampilkan", "daftar", "all"]):
        return "entity_list"
    toks = ql.split()
    if len(toks) <= 4 and any(t in ql for t in ["customer", "pelanggan", "product", "order"]):
        return "entity_list"
    return "other"

def score_tables_from_reranked(reranked: List[Dict[str, Any]], query_tokens: List[str]) -> Dict[str, float]:
    scores: Dict[str, float] = defaultdict(float)
    qt = [t.lower() for t in query_tokens]
    for it in reranked:
        md = it.get("meta") or {}
        tbl = (md.get("table") or md.get("tableName") or "").strip()
        if not tbl:
            continue
        desc = (md.get("description") or "").lower()
        base = 0.0
        if it.get("sparse"):
            base += 0.5
        if tbl.lower() in " ".join(qt):
            base += 1.0
        for t in qt:
            if t and t in desc:
                base += 0.3
        scores[tbl] += base
    return scores

# =========================
# === RETRIEVE (PURE CHROMA)
# =========================
def retrieve_node(state: RAGState) -> RAGState:
    col = build_vector_store()
    q = (state["question"] or "").strip()
    q_l = q.lower()

    qtype = classify_query(q)

    wants_customer_no_id = (
        any(w in q_l for w in ["customer", "customers", "pelanggan"])
        and not re.search(r"\b(id|number|nomor|no|code|kode)\b", q_l)
        and "customer number" not in q_l
    )

    jsonl_paths = state.get("jsonl_paths", [])
    sparse_query_clean = build_sparse_query_from_question(q)
    sparse_tokens = _tokenize_bm25_query(q)
    sparse_field_urns = jsonl_sparse_search(q, jsonl_paths, max_hits=SPARSE_MAX_HITS)
    # --- pretty print untuk hasil SPARSE ---
    sparse_docs_pretty: List[str] = []
    try:
        if sparse_field_urns:
            got = col.get(ids=sparse_field_urns, include=["metadatas"])
            ids   = (got.get("ids") or [])
            metas = (got.get("metadatas") or [])
            for _id, md in zip(ids, metas):
                md = md or {}
                tbl = (md.get("table") or md.get("tableName") or "") or "?"
                c   = (md.get("column") or md.get("columnName") or "") or "?"
                sparse_docs_pretty.append(f"table={tbl}  column={c}")
    except Exception as e:
        # fallback: kalau gagal ambil meta dari Chroma, tampilkan ID-nya saja
        sparse_docs_pretty = list(sparse_field_urns or [])
        if RETRIEVE_DEBUG:
            print(f"[WARN] Sparse meta fetch failed: {e!r}")
            
    sparse_has_customername = any(
        u.lower().endswith("::customername") and ".customers," in u.lower()
        for u in sparse_field_urns
    )

    base_k = max(12, TOP_K)
    dense_k = base_k
    if wants_customer_no_id and not sparse_has_customername:
        dense_k = max(28, base_k)

    query_text = build_dynamic_probe(q)
    res = col.query(
        query_texts=[query_text],
        n_results=dense_k,
        include=["documents", "metadatas", "distances"],
    )

    dense_docs   = (res.get("documents")  or [[]])[0]
    dense_metas  = (res.get("metadatas")  or [[]])[0]
    dense_dists  = (res.get("distances")  or [[]])[0]
    dense_ids    = (res.get("ids")        or [[]])[0]

    n = min(
        len(dense_docs),
        len(dense_metas),
        len(dense_dists) if dense_dists else len(dense_docs),
        len(dense_ids) if dense_ids else len(dense_docs),
    )
    dense_docs  = dense_docs[:n]
    dense_metas = dense_metas[:n]
    dense_dists = (dense_dists or [0.0] * n)[:n]
    dense_ids   = (dense_ids or [""] * n)[:n]

    dense_has_customername = any(
        ((md.get("table") or md.get("tableName") or "").lower() == "customers")
        and ((md.get("column") or md.get("columnName") or "").lower() in ("customername", "customer_name"))
        for md in dense_metas
    )

    scored = []
    for rank, (doc_text, md, dist, _id) in enumerate(zip(dense_docs, dense_metas, dense_dists, dense_ids)):
        base = -float(dist) if dist is not None else 0.0
        sparse_hit = _doc_matches_sparse(md, sparse_field_urns)

        table_name = (md.get("table") or md.get("tableName") or "").strip()
        col_name   = (md.get("column") or md.get("columnName") or "").strip().lower()

        # parse roles
        roles_raw = md.get("tags_role") or "[]"
        roles: List[str] = []
        if isinstance(roles_raw, str):
            try:
                parsed = json.loads(roles_raw)
                if isinstance(parsed, list):
                    roles = [str(r).lower() for r in parsed]
                else:
                    roles = [str(roles_raw).lower()]
            except Exception:
                roles = [str(roles_raw).lower()]
        elif isinstance(roles_raw, list):
            roles = [str(r).lower() for r in roles_raw]

        # parse aliases
        aliases_raw = md.get("tags_alias") or "[]"
        aliases: List[str] = []
        if isinstance(aliases_raw, str):
            try:
                parsed_a = json.loads(aliases_raw)
                if isinstance(parsed_a, list):
                    aliases = [str(a).lower() for a in parsed_a]
                else:
                    aliases = [str(aliases_raw).lower()]
            except Exception:
                aliases = [str(aliases_raw).lower()]
        elif isinstance(aliases_raw, list):
            aliases = [str(a).lower() for a in aliases_raw]

        boosted = base + (SPARSE_BONUS if sparse_hit else 0.0)
        boosted += apply_column_priority_boost(
            q_l=q_l,
            table=table_name,
            col_name=col_name,
            roles=roles,
            aliases=aliases,
        )

        scored.append({
            "rank": rank,
            "id": _id,
            "doc": doc_text,
            "meta": md or {},
            "dist": float(dist) if dist is not None else 0.0,
            "base": base,
            "sparse": sparse_hit,
            "boosted": boosted,
        })

    # auto-inject sangat terbatas
    if wants_customer_no_id and sparse_has_customername and not dense_has_customername:
        sparse_customername_urn = None
        for u in sparse_field_urns:
            if u.lower().endswith("::customername"):
                sparse_customername_urn = u
                break
        if sparse_customername_urn:
            present_ids = {item["id"] for item in scored if item.get("id")}
            if sparse_customername_urn not in present_ids:
                got = col.get(ids=[sparse_customername_urn])
                if got and got.get("ids"):
                    md  = (got.get("metadatas") or [{}])[0]
                    doc = (got.get("documents") or [""])[0]

                    roles_raw = md.get("tags_role") or "[]"
                    roles: List[str] = []
                    if isinstance(roles_raw, str):
                        try:
                            parsed = json.loads(roles_raw)
                            if isinstance(parsed, list):
                                roles = [str(r).lower() for r in parsed]
                            else:
                                roles = [str(roles_raw).lower()]
                        except Exception:
                            roles = [str(roles_raw).lower()]
                    elif isinstance(roles_raw, list):
                        roles = [str(r).lower() for r in roles_raw]

                    aliases_raw = md.get("tags_alias") or "[]"
                    aliases: List[str] = []
                    if isinstance(aliases_raw, str):
                        try:
                            parsed_a = json.loads(aliases_raw)
                            if isinstance(parsed_a, list):
                                aliases = [str(a).lower() for a in parsed_a]
                            else:
                                aliases = [str(aliases_raw).lower()]
                        except Exception:
                            aliases = [str(aliases_raw).lower()]
                    elif isinstance(aliases_raw, list):
                        aliases = [str(a).lower() for a in aliases_raw]

                    table_name = (md.get("table") or md.get("tableName") or "").strip()
                    col_name   = (md.get("column") or md.get("columnName") or "").strip().lower()

                    max_boost = max((x["boosted"] for x in scored), default=0.0)
                    boosted = max_boost + 0.05
                    boosted += apply_column_priority_boost(
                        q_l=q_l,
                        table=table_name,
                        col_name=col_name,
                        roles=roles,
                        aliases=aliases,
                    )

                    scored.append({
                        "rank": len(scored),
                        "id": sparse_customername_urn,
                        "doc": doc,
                        "meta": md or {},
                        "dist": 0.0,
                        "base": 0.0,
                        "sparse": True,
                        "boosted": boosted,
                    })

    reranked = sorted(scored, key=lambda x: x["boosted"], reverse=True)
    reranked = reranked[:MAX_CTX_DOCS]

    if RETRIEVE_DEBUG:
        line = "─" * 150
        print(f"\n{line}")
        print("RETRIEVE DEBUG".center(150))
        print(line)

        print(f"QUESTION      : {q}")
        print(f"DENSE QUERY   : {query_text}")
        print(f"SPARSE QUERY  : {sparse_query_clean}")
        print(f"TOKENS        : {sparse_tokens}\n")

        # =========================
        # SPARSE CANDIDATES
        # =========================
        total_sparse = len(sparse_docs_pretty)
        max_show = min(7, total_sparse)
        print(f"SPARSE CANDIDATES (top {max_show} of {total_sparse}):")
        if total_sparse == 0:
            print("  <none>\n")
        else:
            # prefix row: "  [01] " = 7 chars → header juga 7 spasi
            print(f"{'':7}{'table':<14} {'column':<30}")
            for i, s in enumerate(sparse_docs_pretty[:max_show], start=1):
                # s format: "table=xxx  column=yyy"
                try:
                    _, rest = s.split("table=", 1)
                    table_part, col_part = rest.split("column=", 2)
                    table = table_part.strip() or "?"
                    column = col_part.strip() or "?"
                except Exception:
                    table = "?"
                    column = s
                print(f"  [{i:02d}] {table:<14} {column:<30}")
            if total_sparse > max_show:
                remaining = total_sparse - max_show
                print(f"  ... (+{remaining} more)")
            print()

        # =========================
        # DENSE CANDIDATES
        # =========================
        total_dense = len(dense_metas)
        max_show = min(7, total_dense)
        print(f"DENSE CANDIDATES (top {max_show} of {total_dense}):")
        if total_dense == 0:
            print("  <none>\n")
        else:
            print(f"{'':7}{'table':<14} {'column':<30}")
            for i in range(max_show):
                md  = dense_metas[i] or {}
                tbl = (md.get("table") or md.get("tableName") or "?").strip()
                col = (md.get("column") or md.get("columnName") or "?").strip()
                print(f"  [{i+1:02d}] {tbl:<14} {col:<30}")
            if total_dense > max_show:
                remaining = total_dense - max_show
                print(f"  ... (+{remaining} more)")
            print()

        # =========================
        # RERANKED RESULT
        # =========================
        total_rerank = len(reranked)
        max_show = min(7, total_rerank)
        print(f"RERANKED RESULT (top {max_show} of {total_rerank}):")
        if total_rerank == 0:
            print("  <no reranked candidates>")
        else:
            print(f"{'':7}{'table':<14} {'column':<30} {'sparse':>8}")
            for i, item in enumerate(reranked[:max_show], start=1):
                md   = item.get("meta") or {}
                tbl  = (md.get("table") or md.get("tableName") or "?").strip()
                col  = (md.get("column") or md.get("columnName") or "?").strip()
                sparse_flag = "✓" if item.get("sparse") else "✗"
                print(f"  [{i:02d}] {tbl:<14} {col:<30} {sparse_flag:>8}")
            if total_rerank > max_show:
                remaining = total_rerank - max_show
                print(f"  ... (+{remaining} more)")
        print(f"{line}\n")

    # build allowed + alias_map
    allowed   = defaultdict(list)
    alias_map = defaultdict(dict)
    for item in reranked:
        md = item["meta"] or {}
        t  = (md.get("table") or md.get("tableName") or "").strip()
        c  = (md.get("column") or md.get("columnName") or "").strip()
        if t and c:
            allowed[t].append(c)
            aliases2 = md.get("aliases") or md.get("tags_alias") or md.get("synonyms") or []
            if isinstance(aliases2, str):
                try:
                    maybe = json.loads(aliases2)
                    if isinstance(maybe, list):
                        aliases2 = maybe
                    else:
                        aliases2 = [aliases2]
                except Exception:
                    aliases2 = [aliases2]
            for al in aliases2:
                al = str(al).strip()
                if al:
                    alias_map[t][al.lower()] = c

    token_for_score = _tokenize_bm25_query(q)
    table_scores = score_tables_from_reranked(reranked, token_for_score)

    if classify_query(q) == "entity_list" and wants_customer_no_id:
        filtered_allowed = {t: cols for t, cols in allowed.items() if t.lower() == "customers"}
    else:
        if table_scores:
            sorted_tbls = sorted(table_scores.items(), key=lambda x: x[1], reverse=True)
            top_tables = {name for name, score in sorted_tbls[:2] if score > 0.0}
            if top_tables:
                filtered_allowed = {t: cols for t, cols in allowed.items() if t in top_tables}
            else:
                filtered_allowed = dict(allowed)
        else:
            filtered_allowed = dict(allowed)

    if wants_customer_no_id:
        for t_key in list(filtered_allowed.keys()):
            if t_key.lower() == "customers":
                if "customerName" not in filtered_allowed[t_key]:
                    filtered_allowed[t_key].append("customerName")

    allowed_lines = []
    for t, cols in filtered_allowed.items():
        uniq = sorted(set(cols))
        allowed_lines.append(f"- {t}: {', '.join(uniq)}")
    allowed_block = "### ALLOWED IDENTIFIERS\n" + ("\n".join(allowed_lines) if allowed_lines else "- (none)")

    cards: List[str] = []
    present = set()
    for it in reranked:
        md = it["meta"] or {}
        table = (md.get("table") or md.get("tableName") or "").strip()
        column = (md.get("column") or md.get("columnName") or "").strip()
        if table not in filtered_allowed:
            continue
        desc = (md.get("description") or "").strip()
        if len(desc) > 160:
            desc = desc[:160] + "..."
        cards.append(f"- table: {table} | column: {column} | desc: {desc}")
        present.add((table.lower(), column.lower()))

    injected_cards = []
    for tbl, cols in filtered_allowed.items():
        for col in cols:
            key = (tbl.lower(), col.lower())
            if key not in present:
                short_desc = ""
                if tbl.lower() == "customers" and col.lower() == "customername":
                    short_desc = "customer display name / company name"
                injected_cards.append(f"- table: {tbl} | column: {col} | desc: {short_desc}")
                present.add(key)

    cards = injected_cards + cards
    cards_text = "\n".join(cards)

    context = (allowed_block + "\n\n" + cards_text).strip()
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# =========================
# === SQL POSTPROCESS =====
# =========================
_JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{_JOIN_KW}\b", sql, flags=re.I) is not None

def _should_use_backtick(identifier: str) -> bool:
    if not identifier: return False
    clean = identifier.strip('`"')
    if not clean: return False
    first_char = clean[0]
    if first_char.isupper() or first_char.isdigit():
        return True
    if not re.match(r'^[a-z_][a-z0-9_]*$', clean, re.I):
        return True
    return False

def _smart_backtick(identifier: str) -> str:
    clean = identifier.strip('`"')
    return f"`{clean}`" if _should_use_backtick(clean) else clean

def _split_qualified(name: str) -> List[str]:
    return [p.strip('`" ') for p in name.split(".") if p.strip("\"` ")]

def _fix_bad_backticks(sql: str) -> str:
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return _smart_backtick(inside)
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(_smart_backtick(p) for p in parts)
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`",
                 lambda m: f"{_smart_backtick(m.group(1))}."
                           + ".".join(_smart_backtick(p) for p in _split_qualified(m.group(2))), sql)
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)",
                 lambda m: ".".join(_smart_backtick(p) for p in _split_qualified(m.group(1)))
                           + f".{_smart_backtick(m.group(2))}", sql)
    sql = re.sub(r'`([^`]+?)`(?!\s*\.)', lambda m: _smart_backtick(m.group(1)), sql, flags=re.I)
    return sql

def _explicit_limit_from_question(question: str) -> Optional[int]:
    q = (question or "").lower()
    m = re.search(r"\blimit\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\btop\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\b(show|first|tampilkan|ambil)\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(2)), 1_000_000))
    return None

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    n = _explicit_limit_from_question(question)
    if n is None:
        return sql
    s = sql.rstrip().rstrip(";")
    return s + f" LIMIT {n};"

def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    hard_signals = [
        "distinct",
        "no duplicate",
        "without duplicates",
        "tanpa duplikat",
        "tanpa duplicate",
        "hanya yang unik",
        "dedup",
        "de-dup",
    ]
    if any(s in q for s in hard_signals):
        return True
    if "unique" in q and ("rows" in q or "records" in q or "result" in q):
        return True
    return False

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    if not sql:
        return sql
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)
    return _remove_distinct(sql)

def _fix_missing_space_before_clauses(sql: str) -> str:
    sql = re.sub(r"\s+(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)", " ", sql, flags=re.I)
    sql = re.sub(
        r"([)\],;`])(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)",
        r"\1 ",
        sql,
        flags=re.I,
    )
    sql = re.sub(r"([A-Za-z0-9_\.])(?=(LIMIT|WHERE|HAVING)\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=ORDER\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=GROUP\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"\s+(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b", r" \1", sql, flags=re.I)
    return sql

def _collapse_spaces_outside_quotes(s: str) -> str:
    parts = re.split(r"('(?:''|[^'])*')", s)
    for i in range(len(parts)):
        if i % 2 == 0:
            parts[i] = re.sub(r"[ \t]{2,}", " ", parts[i])
    return "".join(parts)

def _extract_order_by_clause(sql: str) -> Optional[re.Match]:
    return re.search(r"\border\s+by\s+(.*?)(?=(\blimit\b|;|$))", sql, flags=re.I | re.S)

def _remove_order_by_clause(sql: str) -> str:
    return re.sub(r"\s+order\s+by\s+.*?(?=(\blimit\b|;|$))", "", sql, flags=re.I | re.S)

def _question_requests_ordering(question: str) -> bool:
    q = (question or "").lower()
    keywords = [
        "order by", "sort", "sorted", "ascending", "descending", "asc", "desc",
        "alphabetical", "alphabetically", "latest", "newest", "oldest",
        "highest", "lowest", "top", "bottom", "most expensive", "cheapest",
        "urut", "urutkan", "diurutkan", "menaik", "menurun",
        "terbesar", "terkecil", "tertinggi", "terendah", "terbaru", "terlama",
        "termahal", "termurah",
    ]
    return any(k in q for k in keywords)

def _question_prefers_customer_name(question: str) -> bool:
    q = (question or "").lower()
    q = re.sub(r"\b(total\s+)?number\s+of\b", "", q)
    mention_customer = any(w in q for w in ["customer", "customers", "pelanggan"])
    mention_id_like = re.search(r"\b(id|number|nomor|no|code|kode)\b", q) or "customer number" in q
    return mention_customer and not mention_id_like

def _prefer_name_over_number_for_customers(sql: str, question: str) -> str:
    if not _question_prefers_customer_name(question):
        return sql

    s_new = re.sub(
        r"SELECT\s+c\.customerNumber(\s+AS\s+customerNumber)?",
        "SELECT c.customerName AS customerName",
        sql,
        flags=re.I,
    )

    s_new = re.sub(
        r"SELECT\s+c\.customerNumber\s*,\s*COUNT\(",
        "SELECT c.customerName, COUNT(",
        s_new,
        flags=re.I,
    )

    s_new = re.sub(
        r"GROUP BY\s+c\.customerNumber\b",
        "GROUP BY c.customerName",
        s_new,
        flags=re.I,
    )

    return s_new

def _strip_unneeded_count_distinct(sql: str) -> str:
    low = sql.lower()
    if "from customers" in low and "join orders" in low:
        sql = re.sub(
            r"COUNT\s*\(\s*DISTINCT\s+o\.orderNumber\s*\)",
            "COUNT(o.orderNumber)",
            sql,
            flags=re.IGNORECASE,
        )
    return sql

def _postprocess_sql(sql: str, question: str, context: str = "") -> str:
    if not sql:
        return sql
    s = sql.strip()

    def _norm_kw(text: str) -> str:
        CHUNKS = re.compile(
            r"(`[^`]*`|\"([^\"\\]|\\.)*\"|'([^'\\]|\\.)*')",
            re.DOTALL,
        )
        KEYWORDS = [
            "select","from","where","group by","order by","having","limit",
            "inner join","left join","right join","full join","join","on",
        ]

        def normalize_code_segment(seg: str) -> str:
            for kw in KEYWORDS:
                pattern = rf"(?<![A-Za-z0-9_]){kw}(?![A-Za-z0-9_])"
                seg = re.sub(pattern, lambda m: m.group(0).upper(), seg, flags=re.I)
            seg = re.sub(r"[ \t]+", " ", seg)
            return seg

        out, i = [], 0
        for m in CHUNKS.finditer(text):
            out.append(normalize_code_segment(text[i:m.start()]))
            out.append(m.group(0))
            i = m.end()
        out.append(normalize_code_segment(text[i:]))

        return "".join(out)

    s = _norm_kw(s)
    if not s.endswith(";"):
        s += ";"

    s = _fix_bad_backticks(s)
    s = _ensure_reasonable_distinct(s, question)
    s = _prefer_name_over_number_for_customers(s, question)
    ob_match = _extract_order_by_clause(s)
    if ob_match and not _question_requests_ordering(question):
        s = _remove_order_by_clause(s)

    s = _maybe_add_limit(question, s)
    s = _fix_missing_space_before_clauses(s)
    s = _collapse_spaces_outside_quotes(s)
    s = _strip_unneeded_count_distinct(s)
    return s

# ---------- Whitelist parsing + alias repair ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _extract_tables_from_sql(sql: str) -> set:
    tables = set()
    for m in re.finditer(r"\bfrom\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    for m in re.finditer(r"\bjoin\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    qualified = re.findall(
        r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?",
        sql,
    )
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)

    table_names = _extract_tables_from_sql(sql)
    for t in table_names:
        out.setdefault(t, set())

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right",
        "inner","on","and","or","not","as","distinct","count","sum","avg","min","max",
        "having","asc","desc","like","in","between","is","null","case","when","then",
        "else","end"
    }

    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        if tok in table_names:
            continue
        out.setdefault(None, set()).add(tok)

    return out

def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    allowed = _parse_allowed_identifiers(ctx)
    used = _identify_used_identifiers(sql)
    errs: List[str] = []

    explicit_tables = {t for t in used.keys() if t is not None}

    single_table: Optional[str] = None
    if len(explicit_tables) == 1:
        single_table = next(iter(explicit_tables))
    elif len(explicit_tables) == 0 and len(allowed) == 1:
        single_table = next(iter(allowed.keys()))

    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")

    if single_table:
        allowed_cols = allowed.get(single_table, set())
        for col in used.get(None, set()):
            if col not in allowed_cols:
                errs.append(f"{single_table} -> invalid column (unqualified): {col}")

    return errs

def _repair_with_aliases(sql: str, ctx: str) -> str:
    """
    Perbaiki nama kolom berdasar alias_map dan fallback classicmodels,
    TAPI jangan menyentuh teks dalam kutip ('...', "...", `...`)
    supaya alias/label buatan LLM tetap utuh.
    """
    alias_map_per_table = _extract_internal_hint(ctx)

    # global map (lower → canonical)
    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in (alias_map_per_table or {}).items():
        for al, canon in (amap or {}).items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None

    # fallback untuk classicmodels
    classic_fallback = {
        "products": {
            "name": "productName",
            "productname": "productName",
            "listprice": "MSRP",
        },
        "orderdetails": {
            "listprice": "priceEach",
            "price": "priceEach",
        },
        "customers": {
            "name": "customerName",
            "customername": "customerName",
            "id": "customerNumber",
            "customerid": "customerNumber",
            "customer_id": "customerNumber",
            "id_customer": "customerNumber",
            "kodecustomer": "customerNumber",
            "kode_pelanggan": "customerNumber",
        },
    }
    for _t, amap in classic_fallback.items():
        for al, canon in amap.items():
            al = al.lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] is None:
                pass

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right","inner",
        "on","and","or","not","as","distinct","count","sum","avg","min","max","having",
        "asc","desc","like","in","between","is","null","case","when","then","else","end"
    }
    token_pat = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")

    def replace_in_code_segment(seg: str) -> str:
        def smart_repl(m):
            w = m.group(0)
            lw = w.lower()
            if lw in keywords:
                return w
            canon = global_map.get(lw)
            if not canon:
                return w
            if not _ALNUM.match(canon):
                return w
            return canon
        return token_pat.sub(smart_repl, seg)

    parts = re.split(r"('(?:''|[^'])*'|\"(?:\"\"|[^\"])*\"|`[^`]*`)", sql)
    out_parts = []
    for idx, part in enumerate(parts):
        if idx % 2 == 1:
            out_parts.append(part)  # quoted → jangan disentuh
        else:
            out_parts.append(replace_in_code_segment(part))  # code → boleh direpair
    return "".join(out_parts)

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.
If the query is about customers and the user does NOT explicitly ask for an id/number/code, prefer customerName over customerNumber AND also prefer customerName over contactFirstName/contactLastName when both are available in ALLOWED IDENTIFIERS.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

def _sanitize_notes(sql: str, notes: str) -> str:
    if not notes:
        return notes
    if not _has_join(sql or ""):
        return "No joins used."
    return notes.strip()

# ---------- extra helpers for generation robustness ----------
def _try_llm_json_three_times(llm: ChatOllama, base_prompt: str) -> Optional[Dict[str, Any]]:
    prompts = [
        base_prompt,
        base_prompt + "\n\nReturn ONLY the JSON object. Do not add explanation.",
        base_prompt + "\n\nReturn ONLY JSON with keys \"sql\" and \"notes\". No extra text. Example: {\"sql\":\"SELECT 1;\",\"notes\":\"No joins used.\"}",
    ]
    for p in prompts:
        resp = llm.invoke(p)
        content = getattr(resp, "content", None) or str(resp)
        try:
            obj = json.loads(content)
            if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                return obj
        except Exception:
            m = re.search(r"\{.*\}", content, flags=re.S)
            if m:
                try:
                    obj = json.loads(m.group(0))
                    if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                        return obj
                except Exception:
                    pass
    return None

def _heuristic_sql_from_question(question: str, ctx: str) -> Optional[str]:
    q = (question or "").lower()
    if "customer" in q and "order" in q and ("total" in q or "number of" in q):
        return (
            "SELECT c.customerName, COUNT(o.orderNumber) AS totalOrders\n"
            "FROM customers c\n"
            "LEFT JOIN orders o ON o.customerNumber = c.customerNumber\n"
            "GROUP BY c.customerName\n"
            "ORDER BY totalOrders DESC;"
        )
    return None

# ---------- Generation (with validation & regeneration) ----------
def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        seed=42
    )
    ctx = state.get("context", "")
    question = state.get("question", "")
    base_prompt = _make_generation_prompt(ctx, question)

    answer_out = ""
    draft_sql = ""
    final_sql = ""
    notes = ""

    # <<< ADD: timer untuk total waktu LLM
    t_llm_total_ms = 0
    def _since_ms(t0: float) -> int:
        return int((perf_counter() - t0) * 1000)

    # panggilan LLM pertama
    t0 = perf_counter()
    obj = _try_llm_json_three_times(llm, base_prompt)
    t_llm_total_ms += _since_ms(t0)

    if obj is not None:
        draft_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()
    else:
        heuristic = _heuristic_sql_from_question(question, ctx)
        if heuristic:
            draft_sql = heuristic
            notes = "Heuristic fallback for customers + orders."
        else:
            draft_sql = "SELECT 1;"
            notes = "Heuristic trivial fallback. No joins used."

    if draft_sql:
        _dbg_print_two("[SQL] Generated (before post-processing):", draft_sql)

    # 1) postprocess awal
    final_sql = _postprocess_sql(draft_sql, question, ctx) if draft_sql else draft_sql

    # 2) repair alias tapi aman terhadap quoted alias
    if final_sql:
        final_sql = _repair_with_aliases(final_sql, ctx)

    # validasi allowed identifiers
    violations = _validate_identifiers(final_sql or "", ctx)
    if violations:
        force_msg = (
            "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
            + "\n- ".join(violations)
            + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
              "If a synonym appears, choose the canonical column."
        )
        strict_prompt = _make_generation_prompt(ctx + "\n\n" + force_msg, question)

        # <<< ADD: ukur waktu LLM untuk strict re-generate
        t1 = perf_counter()
        obj2 = _try_llm_json_three_times(llm, strict_prompt)
        t_llm_total_ms += _since_ms(t1)

        if obj2 is not None:
            draft2 = (obj2.get("sql") or "").strip()
            notes2 = (obj2.get("notes") or "").strip()
        else:
            draft2 = ""
            notes2 = ""
        sql2 = _postprocess_sql(draft2 or "", question, ctx) if draft2 else ""
        if sql2:
            sql2 = _repair_with_aliases(sql2, ctx)
            sql2 = _postprocess_sql(sql2, question, ctx)
        v1 = len(violations)
        v2 = len(_validate_identifiers(sql2 or "", ctx))
        if v2 <= v1 and sql2:
            final_sql, notes = sql2, (notes2 or notes)

    notes = _sanitize_notes(final_sql or "", notes or "")

    if final_sql:
        answer_out = final_sql
    else:
        answer_out = draft_sql or "SELECT 1;"

    state["answer"] = answer_out
    state["draft_sql"] = draft_sql
    state["final_sql"] = final_sql
    state["llm_ms"] = int(t_llm_total_ms)  # <<< ADD
    return state

# =========================
# === APP GRAPH ===========
# =========================
def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("No tables found.")
        return
    print("\nAvailable tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))

    if preview_rows and preview_cols:
        print("\nQuery Result (preview):")
        print(
            tabulate(
                preview_rows,
                headers=preview_cols,
                tablefmt="github",
                showindex=range(1, len(preview_rows) + 1),
            )
        )
        print(
            f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
            f"(full result: {total_rows} rows × {total_cols} columns)"
        )
    else:
        print("Query executed but returned no rows.")

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        first_q = input("Ask me anything about your data: ").strip()
        if not first_q:
            print("No question entered. Exiting.")
            return

        t0_total = _now()  # start total timing untuk first run

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
            "draft_sql": "",
            "final_sql": "",
        }

        # Jalankan ingest dulu biar bisa tampilkan report sebelum retrieve debug
        ingest_state = ingest_node(init)

        col = build_vector_store()
        total = col.count()
        line = "─" * 150
        print(f"\n{line}")
        print("Collection Ingestion Report".center(150))
        print(line)
        print(f"Added (first run): {ingest_state.get('added', 0)}")
        print(f"Updated (upserted): {ingest_state.get('updated', 0)}")
        print(f"Skipped (unchanged): {ingest_state.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")
        print(f"{line}\n")

        # Lanjut ke retrieve + generate
        state = retrieve_node(ingest_state)
        state = generate_node(state)

        async def exec_via_mcp(sql_block: str, final_sql: Optional[str] = None):
            # sql_block sekarang biasanya sudah hanya berisi SQL,
            # tapi tetap split untuk backward compatibility
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = (final_sql or sql_only).rstrip(";").strip()

            # Log singkat & jelas: cuma query final yang dieksekusi
            print("\n[SQL EXEC] Final query sent to MCP:")
            print(sql_clean + ";")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

        _log_total("End-to-end total time (first run)", t0_total)

        print("\nAsk me more questions! (type 'exit' to quit')")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            t0_total = _now()

            state = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
                "draft_sql": "",
                "final_sql": "",
            }
            state = retrieve_node(state)
            state = generate_node(state)

            await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

            _log_total("End-to-end total time (this question)", t0_total)

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()

## **CHECK COLLECTION**

In [ ]:
from pprint import pprint
import chromadb
import numpy as np  # ok if installed; fallback guards below handle absence

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(COLLECTION_NAME)

print("== BASIC INFO ==")
print("name :", col.name)
print("count:", col.count())

print("\n== FIRST 5 DOCS (with embeddings) ==")
res = col.get(limit=5, include=["metadatas", "documents", "embeddings"])

ids = res.get("ids", [])

embs = res.get("embeddings", None)
# normalize to a Python list of vectors or None
if embs is None:
    embs = []
else:
    # if it's a numpy array, convert to list; otherwise leave as-is
    try:
        # check without importing numpy explicitly
        import numpy as _np  # noqa
        if hasattr(embs, "tolist"):
            embs = embs.tolist()
    except Exception:
        # best-effort: if it still behaves like an array, wrap in list()
        try:
            embs = list(embs)
        except Exception:
            pass

n = min(len(ids), len(embs)) if embs else len(ids)

for i in range(n):
    id_ = ids[i]
    emb = embs[i] if i < len(embs) else None
    dim = None if emb is None else len(emb)
    print(f"{i+1:>2}. {id_} | emb_dim={dim}")

# count how many are None (only for the portion we fetched)
none_count = sum(1 for i in range(min(len(ids), len(embs))) if embs[i] is None) if embs else 0
print("Embeddings None (from first batch):", none_count)

# Optional: detail of the first doc
if ids:
    got_one = col.get(ids=[ids[0]], include=["metadatas", "documents", "embeddings"])
    print("\n== DETAIL DOC PERTAMA ==")
    pprint({k: got_one[k] for k in got_one if k in ("ids","metadatas","documents")})
    e0 = got_one.get("embeddings", [None])[0]
    if hasattr(e0, "tolist"):
        e0 = e0.tolist()
    print("emb_dim (doc[0]):", None if e0 is None else len(e0))

## **DELETE COLLECTION**

In [ ]:
import chromadb

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# cek dulu ada atau nggak
try:
    client.get_collection(COLLECTION_NAME)  # kalau gak ada bakal raise
    client.delete_collection(COLLECTION_NAME)
    print(f"[OK] collection '{COLLECTION_NAME}' deleted.")
except Exception as e:
    print(f"[INFO] collection '{COLLECTION_NAME}' not found or already deleted. ({e})")